In [0]:
select * from datawerhouse1.silver.customers_txt;
create or replace table datawerhouse1.gold.customers_txt as 
select 
customer_id,
credit_term_days,
annual_revenue_potential,
customer_name,
customer_type,
primary_freight_type,
account_status,
contract_start_date
from datawerhouse1.silver.customers_txt;
select * from datawerhouse1.gold.customers_txt


In [0]:
-- This query returns the top customer_name by annual_revenue_potential for each customer_type.
select
  customer_name,
  annual_revenue_potential,
  customer_type
from (
  select
    customer_name,
    annual_revenue_potential,
    customer_type,
    row_number() over (partition by customer_type order by annual_revenue_potential desc) as rn
  from datawerhouse1.gold.customers_txt
)
where rn = 1

In [0]:
-- Returns the customer with the highest credit_term_days, prioritizing by annual_revenue_potential in case of ties.
select
  customer_id,
  customer_name,
  customer_type,
  credit_term_days,
  annual_revenue_potential
from datawerhouse1.gold.customers_txt
order by credit_term_days desc, annual_revenue_potential desc
limit 1

In [0]:
  -- Returns the top 5 primary_freight_type by total annual_revenue_potential, including earliest contract_start_date.
select
  any_value(customer_name) as customer_name,
  primary_freight_type,
  sum(annual_revenue_potential) as total_revenue,
  min(contract_start_date) as earliest_contract_start_date
from datawerhouse1.gold.customers_txt
group by primary_freight_type
order by total_revenue desc
limit 5

In [0]:
--return bottom 5 primary_freight_type by total annual_revenue_potential, perfectly ordered from smallest to largest.
select
  any_value(customer_name) as customer_name,
  primary_freight_type,
  sum(annual_revenue_potential) as total_revenue
from datawerhouse1.gold.customers_txt
group by primary_freight_type
order by total_revenue asc
limit 5

In [0]:
select * from datawerhouse1.silver.delivery_events;
create or replace table datawerhouse1.gold.delivery_events as 
select 
event_id,
load_id,
trip_id,
event_type,
facility_id,
scheduled_datetime,
actual_datetime,
delay_minutes,
detention_minutes,
on_time_flag,
location_city,
location_state
from datawerhouse1.silver.delivery_events;
select * from datawerhouse1.gold.delivery_events

In [0]:
select * from datawerhouse1.silver.facilities;
create or replace table datawerhouse1.gold.facilities as 
select 
facility_id,
facility_name,
facility_type,
city as facility_city,
state as facility_state,
latitude as facility_latitude,
longitude as facility_longitude,
dock_doors as facility_dock_doors,
operating_hours as facility_operating_hours
from datawerhouse1.silver.facilities;
select * from datawerhouse1.gold.facilities

In [0]:
select * from datawerhouse1.silver.loads;
create or replace table datawerhouse1.gold.loads as 
select 
load_id,
customer_id,
route_id,
load_date,
load_type,
weight_lbs,
pieces,
revenue,
fuel_surcharge,
accessorial_charges,
load_status,
booking_type
from datawerhouse1.silver.loads;
select * from datawerhouse1.gold.loads

In [0]:
-- Returns joined load, customer, delivery event, and facility details.
select
  i.load_id,
  i.customer_id,
  i.route_id,
  c.customer_name,
  c.customer_type,
  de.event_id,
  de.trip_id,
  f.facility_id,
  f.facility_name
from datawerhouse1.gold.loads i
left join datawerhouse1.gold.customers_txt c on i.customer_id = c.customer_id
left join datawerhouse1.gold.delivery_events de on i.load_id = de.load_id
left join datawerhouse1.gold.facilities f on de.facility_id = f.facility_id;

-- Returns the top 5 primary_freight_type by total annual_revenue_potential, including earliest contract_start_date and scheduled delivery event.
select
  any_value(c.customer_name) as customer_name,
  c.primary_freight_type,
  sum(c.annual_revenue_potential) as total_revenue,
  min(c.contract_start_date) as earliest_contract_start_date,
  min(de.scheduled_datetime) as earliest_scheduled_delivery_event
from datawerhouse1.gold.customers_txt c
left join datawerhouse1.gold.loads l on c.customer_id = l.customer_id
left join datawerhouse1.gold.delivery_events de on l.load_id = de.load_id
group by c.primary_freight_type
order by total_revenue desc
limit 5;

-- Returns the bottom 5 primary_freight_type by total annual_revenue_potential, including earliest contract_start_date and scheduled delivery event.
select
  any_value(c.customer_name) as customer_name,
  c.primary_freight_type,
  sum(c.annual_revenue_potential) as total_revenue,
  min(c.contract_start_date) as earliest_contract_start_date,
  min(de.scheduled_datetime) as earliest_scheduled_delivery_event
from datawerhouse1.gold.customers_txt c
left join datawerhouse1.gold.loads l on c.customer_id = l.customer_id
left join datawerhouse1.gold.delivery_events de on l.load_id = de.load_id
group by c.primary_freight_type
order by total_revenue asc
limit 5;

-- Returns the top 5 primary_freight_type by total annual_revenue_potential, including earliest contract_start_date, earliest scheduled delivery event, and facility city.
select
  any_value(c.customer_name) as customer_name,
  c.primary_freight_type,
  sum(c.annual_revenue_potential) as total_revenue,
  min(c.contract_start_date) as earliest_contract_start_date,
  min(de.scheduled_datetime) as earliest_scheduled_delivery_event,
  min(f.facility_city) as facility_city
from datawerhouse1.gold.customers_txt c
left join datawerhouse1.gold.loads l on c.customer_id = l.customer_id
left join datawerhouse1.gold.delivery_events de on l.load_id = de.load_id
left join datawerhouse1.gold.facilities f on de.facility_id = f.facility_id
group by c.primary_freight_type
order by total_revenue desc
limit 5;

-- Returns the bottom 5 primary_freight_type by total annual_revenue_potential, including earliest contract_start_date, earliest scheduled delivery event, and facility city.
select
  any_value(c.customer_name) as customer_name,
  c.primary_freight_type,
  sum(c.annual_revenue_potential) as total_revenue,
  min(c.contract_start_date) as earliest_contract_start_date,
  min(de.scheduled_datetime) as earliest_scheduled_delivery_event,
  min(f.facility_city) as facility_city
from datawerhouse1.gold.customers_txt c
left join datawerhouse1.gold.loads l on c.customer_id = l.customer_id
left join datawerhouse1.gold.delivery_events de on l.load_id = de.load_id
left join datawerhouse1.gold.facilities f on de.facility_id = f.facility_id
group by c.primary_freight_type
order by total_revenue asc
limit 5;






In [0]:
-- Returns the top 5 load_type by total revenue in round figures.
select
  load_type,
  round(sum(revenue)) as total_revenue
from datawerhouse1.gold.loads
group by load_type
order by total_revenue desc
limit 10

In [0]:
-- Returns the top 5 facility_name by facility_type 'cross dock' according to total annual_revenue_potential  
select
  f.facility_name,
  f.facility_type,
  f.facility_city as location_city,
  f.facility_state,
  f.facility_dock_doors,
  l.load_type,
  sum(c.annual_revenue_potential) as total_revenue
from datawerhouse1.gold.facilities f
left join datawerhouse1.gold.delivery_events de on f.facility_id = de.facility_id
left join datawerhouse1.gold.loads l on de.load_id = l.load_id
left join datawerhouse1.gold.customers_txt c on l.customer_id = c.customer_id
where f.facility_type = 'CROSS-DOCK'
group by f.facility_name, f.facility_type, f.facility_city, f.facility_state, f.facility_dock_doors, l.load_type
order by total_revenue desc
limit 5;


-- Returns the top 5 delivery events by location_city according to total annual_revenue_potential from customer table.
-- Returns the top 5 pieces according to total revenue from loads table.
select
  c.customer_name,
  l.pieces,
  round(sum(l.revenue)) as total_revenue
from datawerhouse1.gold.loads l
left join datawerhouse1.gold.customers_txt c on l.customer_id = c.customer_id
group by c.customer_name, l.pieces
order by total_revenue desc
limit 5;
select
  pieces,
  round(sum(revenue)) as total_revenue
from datawerhouse1.gold.loads
group by pieces
order by total_revenue desc
limit 5;
select
  de.location_city,
  de.event_type,
  sum(c.annual_revenue_potential) as total_revenue
from datawerhouse1.gold.delivery_events de
left join datawerhouse1.gold.loads l on de.load_id = l.load_id
left join datawerhouse1.gold.customers_txt c on l.customer_id = c.customer_id
group by de.location_city, de.event_type
order by total_revenue desc
limit 5;
select
  de.location_city,
  sum(c.annual_revenue_potential) as total_revenue
from datawerhouse1.gold.delivery_events de
left join datawerhouse1.gold.loads l on de.load_id = l.load_id
left join datawerhouse1.gold.customers_txt c on l.customer_id = c.customer_id
group by de.location_city
order by total_revenue desc
limit 5;


In [0]:
SELECT * FROM datawerhouse1.GOLD.loads

In [0]:
SELECT * FROM datawerhouse1.silver.driver_monthly_metrics;
CREATE OR REPLACE TABLE datawerhouse1.gold.driver_monthly_metrics as 
SELECT
  driver_id,
  month,
  trips_completed,
  total_miles,
  total_revenue,
  average_mpg,
  total_fuel_gallons,
  on_time_delivery_rate,
  average_idle_hours
FROM datawerhouse1.silver.driver_monthly_metrics;
select * from datawerhouse1.gold.driver_monthly_metrics

In [0]:
select * from datawerhouse1.silver.drivers;
create or replace table datawerhouse1.gold.drivers as 
select
  driver_id,
  first_name,
  last_name,
  hire_date,
  termination_date,
  license_number,
  license_state,
  date_of_birth,
  home_terminal,
  employment_status,
  cdl_class,
  years_experience
  from datawerhouse1.silver.drivers;
select * from datawerhouse1.gold.drivers;
-- Returns the top 5 active drivers by years of experience.
select
  driver_id,
  concat(first_name, ' ', last_name) as full_name,
  years_experience
from datawerhouse1.gold.drivers
where lower(employment_status) = 'active'
and years_experience is not null
order by years_experience desc
limit 5;
-- Returns the top 5 active drivers by years of experience, grouped by home_terminal.
select
  home_terminal,
  driver_id,
  concat(first_name, ' ', last_name) as full_name,
  years_experience
from datawerhouse1.gold.drivers
where lower(employment_status) = 'active'
and years_experience is not null
order by home_terminal, years_experience desc
limit 5;
select
  home_terminal,
  driver_id,
  concat(first_name, ' ', last_name) as full_name,
  years_experience
from datawerhouse1.gold.drivers
where employment_status = 'Active'
order by home_terminal, years_experience desc
limit 5;
select
  driver_id,
  first_name,
  last_name,
  years_experience
from datawerhouse1.gold.drivers
where lower(employment_status) = 'active'
and years_experience is not null
order by years_experience desc
limit 5;
select
  driver_id,
  first_name,
  last_name,
  years_experience
from datawerhouse1.gold.drivers
where employment_status = 'Active'
order by years_experience desc
limit 5

In [0]:
select * from datawerhouse1.silver.fuel_purchases;
CREATE OR REPLACE TABLE datawerhouse1.gold.fuel_purchases
AS
SELECT
  fuel_purchase_id,
  trip_id,
  truck_id,
  driver_id,
  Is_Known,
  purchase_date,
  city,
  state,
  gallons,
  price_per_gallon,
  total_cost,
  fuel_card_number
FROM datawerhouse1.silver.fuel_purchases;
select * from datawerhouse1.gold.fuel_purchases;



In [0]:
select * from datawerhouse1.silver.maintenance_records;
create or replace table datawerhouse1.gold.maintenance_records as 
select
  maintenance_id,
  truck_id,
  maintenance_date,
  maintenance_type,
  odometer_reading,
  labor_hours,
  labor_cost,
  parts_cost,
  total_cost,
  downtime_hours,
  facility_location,
  service_description
from datawerhouse1.silver.maintenance_records;
select * from datawerhouse1.gold.maintenance_records;


In [0]:
select * from datawerhouse1.silver.safety_incidents;
create or replace table datawerhouse1.gold.safety_incidents as 
select
  safety_incident_id,
  truck_id,
  incident_date,
  incident_type,
  location_city,
  location_state,
  at_fault_flag,
  injury_flag,
  vehicle_damage_cost,
  cargo_damage_cost,
  claim_amount,
  preventable_flag,
  description
  from datawerhouse1.silver.safety_incidents;
select * from datawerhouse1.gold.safety_incidents

In [0]:
    select * from datawerhouse1.silver.trips;
create or replace table datawerhouse1.gold.trips as 
select
  trip_id,
  load_id,
  driver_id,
  truck_id,
  trailer_id,
  dispatch_date,
  actual_distance_miles,
  actual_duration_hours,
  fuel_gallons_used,
  average_mpg,
  idle_time_hours,
  trip_status
from datawerhouse1.silver.trips;
select * from datawerhouse1.gold.trips

In [0]:

create or replace table datawerhouse1.gold.routes as 
select
  route_id,
  origin_city,
  origin_state,
  destination_city,
  destination_state,
  typical_distance_miles,
  base_rate_per_mile,
  fuel_surcharge_rate,
  typical_transit_days
from datawerhouse1.silver.routes;
select * from datawerhouse1.gold.routes
 

In [0]:
create or replace table datawerhouse1.gold.trailers as 
select
  trailer_id,
  trailer_number,
  trailer_type,
  length_feet,
  model_year,
  vin,
  acquisition_date,
  status,
  current_location
  from datawerhouse1.silver.trailers;
select * from datawerhouse1.gold.trailers

In [0]:

create or replace table datawerhouse1.gold.trucks as 
select
  truck_id,
  unit_number,
  make,
  model_year,
  vin,
  acquisition_date,
  acquisition_mileage,
  fuel_type,
  tank_capacity_gallons,
  status,
  home_terminal
  from datawerhouse1.silver.trucks;
select * from datawerhouse1.gold.trucks

In [0]:
create or replace table datawerhouse1.gold.truck_utilization_metrics as 
select
  truck_id,
  month,
  trips_completed,
  total_miles,
  total_revenue,
  average_mpg,
  maintenance_events,
  maintenance_cost,
  downtime_hours,
  utilization_rate
from datawerhouse1.silver.truck_utilization_metrics;
select * from datawerhouse1.gold.truck_utilization_metrics

In [0]:
select * from datawerhouse1.gold.driver_monthly_metrics;
-- Returns the top 5 driver_id by trips_completed, ordered by highest total_revenue
SELECT
  driver_id,
  sum(trips_completed) as trips_completed,
  round(sum(total_revenue)) as total_revenue
FROM datawerhouse1.gold.driver_monthly_metrics
GROUP BY driver_id
ORDER BY trips_completed DESC, total_revenue DESC
LIMIT 5;

In [0]:
select
  fuel_purchase_id,
  trip_id,
  coalesce(truck_id, 'Unknown') as truck_id,
  coalesce(driver_id, 'Unknown') as driver_id,
  Is_Known,
  purchase_date,
  city,
  state,
  gallons,
  price_per_gallon,
  total_cost,
  fuel_card_number
from datawerhouse1.gold.fuel_purchases;



-- Returns the top 5 trucks with the lowest maintenance_events, including their latest fuel purchase date
-- Returns the top 5 trucks with the lowest maintenance_events, ordered by rounded total_revenue, including their latest fuel purchase date
SELECT
  tum.truck_id,
  tum.maintenance_events,
  round(tum.total_revenue) as total_revenue,
  max(fp.purchase_date) as latest_fuel_purchase_date
FROM datawerhouse1.gold.truck_utilization_metrics tum
LEFT JOIN datawerhouse1.gold.fuel_purchases fp
  ON tum.truck_id = fp.truck_id
GROUP BY tum.truck_id, tum.maintenance_events, tum.total_revenue
ORDER BY tum.maintenance_events ASC, total_revenue DESC
LIMIT 5;

SELECT
  tum.truck_id,
  tum.maintenance_events,
  fp.purchase_date

FROM datawerhouse1.gold.truck_utilization_metrics tum
LEFT JOIN datawerhouse1.gold.fuel_purchases fp

  ON tum.truck_id = fp.truck_id 
  GROUP BY tum.truck_id, tum.maintenance_events, fp.purchase_date
ORDER BY tum.maintenance_events ASC, fp.purchase_date DESC
LIMIT 5;


In [0]:
select 
  d.first_name,
  tum.maintenance_events,
  fp.purchase_date
from datawerhouse1.gold.truck_utilization_metrics tum
left join datawerhouse1.gold.fuel_purchases fp on tum.truck_id = fp.truck_id
left join datawerhouse1.gold.drivers d on fp.driver_id = d.driver_id
where d.first_name is not null
order by tum.maintenance_events desc, fp.purchase_date desc
limit 5;

In [0]:
select * from datawerhouse1.gold.safety_incidents;
select 
  fuel_purchase_id,
  trip_id,
  coalesce(fp.truck_id, 'Unknown') as truck_id,
  coalesce(fp.driver_id, 'Unknown') as driver_id,
  Is_Known,
  purchase_date,
  city,
  state,
  gallons,
  price_per_gallon,
  total_cost,
  fuel_card_number,
  si.location_city
from datawerhouse1.gold.fuel_purchases fp
left join datawerhouse1.gold.safety_incidents si
  on fp.truck_id = si.truck_id;
-- Returns the top 5 Customer Complaint and Moving Violations by vehicle_damage_cost with description using CTE
with incident_cte as (
  select
    safety_incident_id,
    truck_id,
    incident_date,
    incident_type,
    vehicle_damage_cost,
    description
  from datawerhouse1.gold.safety_incidents
  where incident_type in ('Customer Complaint', 'Moving Violation')
    and vehicle_damage_cost is not null
)
select
  incident_type,
  vehicle_damage_cost,
  description
from incident_cte
order by vehicle_damage_cost desc
limit 5;
with location_city_cte as (
  select
    location_city,
    incident_date,
    incident_type,
    vehicle_damage_cost,
    description
    from datawerhouse1.gold.safety_incidents
    where location_city in ('COLUMBUS','KANSAS CITY') 
    and vehicle_damage_cost is not null
)
select
  location_city,
  incident_type,
  vehicle_damage_cost,
  description
from location_city_cte
order by vehicle_damage_cost desc
limit 5;
select
  fuel_purchase_id,
  trip_id,
  truck_id,
  driver_id,
  purchase_date,
  city,
  state,
  gallons,
  price_per_gallon,
  total_cost,
  fuel_card_number
from datawerhouse1.gold.fuel_purchases
where lower(city) = 'leawood';

In [0]:
select * from datawerhouse1.gold.trailers;
select * from datawerhouse1.gold.fuel_purchases;
select * from datawerhouse1.gold.trips;

-- Returns the top 5 active trailer status and current_location by total fuel purchase cost, including trip_status using CTE
with trailer_cost_cte as (
  select
    t.trailer_id,
    t.status,
    t.current_location,
   round( sum(fp.total_cost)) as total_cost,
    any_value(tr.trip_status) as trip_status
  from datawerhouse1.gold.trailers t
  left join datawerhouse1.gold.trips tr on t.trailer_id = tr.trailer_id
  left join datawerhouse1.gold.fuel_purchases fp on tr.truck_id = fp.truck_id
  where lower(t.status) = 'active'
  group by t.trailer_id, t.status, t.current_location
)
select
  status,
  current_location,
  total_cost,
  trip_status
from trailer_cost_cte
order by total_cost desc
limit 5;

In [0]:
select * from datawerhouse1.gold.maintenance_records;
select * from datawerhouse1.gold.fuel_purchases;
SELECT * FROM datawerhouse1.gold.drivers;
SELECT * FROM datawerhouse1.gold.safety_incidents;
--top 5 maintenance type by total cost
select 
    maintenance_type,
    round(sum(total_cost)) as tc
from datawerhouse1.gold.maintenance_records
group by maintenance_type
order by tc desc
limit 5;

---find ENGINE according to purchase date by city as part cost BY USING LEFT JOIN
select 
    m.maintenance_type,
    round(SUM(M.parts_cost)) as  PARTSCOST,
    f.purchase_date,
    f.city
from datawerhouse1.gold.maintenance_records m
left join datawerhouse1.gold.fuel_purchases f
on m.truck_id = f.truck_id
where m.maintenance_type = 'ENGINE'
group by m.maintenance_type, f.purchase_date, f.city
order by f.purchase_date desc
limit 5;

---TOTAL ENGINE MAINTENANCE COSTS FOR MOST RECENT PURCHASE DATE BY TRUCK AND COST ALSO RELEATED TO SPECIFIC DRIVERS, LOCATION OF DRIVERS, YEARS OF EXPERIENCE BY USING LEFT JOIN.

SELECT 
   M.maintenance_type,
   round(SUM(M.parts_cost)) as  PARTSCOST,
   F.purchase_date,
   F.city,
   D.first_name,
   D.TERMINATION_DATE,
   D.YEARS_EXPERIENCE
FROM datawerhouse1.gold.maintenance_records M
LEFT JOIN datawerhouse1.gold.fuel_purchases F
ON M.truck_id = F.truck_id
LEFT JOIN datawerhouse1.gold.drivers D
ON F.driver_id = D.driver_id
WHERE M.maintenance_type = 'ENGINE'
GROUP BY M.maintenance_type, F.purchase_date, F.city, D.first_name, D.TERMINATION_DATE, D.YEARS_EXPERIENCE
ORDER BY F.purchase_date DESC
LIMIT 5;
---FIND INCIDINT TYPE BY DRIVER NAME AND LOCATION OF DRIVERS BY USING LEFT JOIN.
SELECT
  SI.safety_incident_id,
  SI.incident_type,
  SI.location_city,
  SI.location_state,
  FP.fuel_purchase_id,
  FP.purchase_date,
  D.first_name,
  D.last_name
FROM datawerhouse1.gold.safety_incidents SI
LEFT JOIN datawerhouse1.gold.fuel_purchases FP
  ON SI.truck_id = FP.truck_id
LEFT JOIN datawerhouse1.gold.drivers D
  ON FP.driver_id = D.driver_id
ORDER BY SI.location_city DESC
LIMIT 5;
SELECT
  SI.safety_incident_id,
  SI.incident_type,
  SI.location_city,
  SI.location_state,
  FP.fuel_purchase_id,
  FP.purchase_date,
  D.first_name,
  D.last_name
FROM datawerhouse1.gold.safety_incidents SI
LEFT JOIN datawerhouse1.gold.fuel_purchases FP
  ON SI.truck_id = FP.truck_id
LEFT JOIN datawerhouse1.gold.drivers D
  ON FP.driver_id = D.driver_id
WHERE upper(SI.location_city) = 'CHICAGO'
AND D.first_name = 'Charles'
AND D.years_experience = '18'
ORDER BY SI.location_city DESC
LIMIT 5;
SELECT
  SI.safety_incident_id,
  SI.incident_type,
  SI.location_city,
  SI.location_state,
  D.first_name,
  D.last_name,
  D.years_experience
FROM datawerhouse1.gold.safety_incidents SI
LEFT JOIN datawerhouse1.gold.fuel_purchases FP
  ON SI.truck_id = FP.truck_id
LEFT JOIN datawerhouse1.gold.drivers D
  ON FP.driver_id = D.driver_id
WHERE D.first_name = 'Charles'
  AND D.years_experience = 18
  AND SI.incident_type = 'Accident'
ORDER BY SI.location_city DESC
LIMIT 5;



